In [97]:
# We'll first have to create training, validation, and test dataset with same distribution.
import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
# I wanna see as much content as possible


amazon= pd.read_csv("outputs/df_amazon.csv",index_col=0,low_memory=False)
goodreads= pd.read_csv("outputs/df_goodreads.csv",index_col=0)
goodreads=goodreads[["isbn","title","author","rating","numRatings","language","genres","bookFormat","edition","pages","publisher","publishDate","price","ISBN_clean"]]
metabooks= pd.read_csv("outputs/df_meta_books_sampled.csv",index_col=0)
metabooks=metabooks[["ISBN_10","title","Author_Name","average_rating","rating_number","Language","categories","Publisher","published_date","Page_Count","price","ISBN_clean"]].reset_index(drop=True)

> Dataset pairs to be used
* metabooks & amazon
* metabooks & goodreads


In [ ]:
def _norm_text(x):
    if pd.isna(x): return ""
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def add_row_id(df: pd.DataFrame, prefix: str, start: int = 1, colname: str | None = None):
    """
    Adds a stable row identifier like 'amazon_1', 'amazon_2', ...
    Inserts it as the FIRST column and returns the modified DataFrame.

    Args:
        df:       Input DataFrame.
        prefix:   String prefix for IDs (e.g., "amazon", "metabooks", "goodreads").
        start:    Starting index (default 1).
        colname:  Name of the ID column; defaults to f"{prefix}_row_id".
    """
    if colname is None:
        colname = f"{prefix}_row_id"
    seq = range(start, start + len(df))
    ids = [f"{prefix}_{i}" for i in seq]
    out = df.copy()
    out.insert(0, colname, ids) 
    return out

def _norm_text(x):
    if pd.isna(x): return ""
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def _pick(df, candidates):
    if isinstance(candidates, str):
        return candidates if candidates in df.columns else None
    for c in candidates:
        if c in df.columns: return c
    return None

def canonicalize_books_rows(
    df: pd.DataFrame,
    *,
    prefix: str,                 # e.g., "amazon", "metabooks", "goodreads"
    title_cols,                  # list of possible title column names
    author_cols,                 # list of possible author column names
    row_id_colname: str | None = None  # if you already have e.g. "amazon_row_id"
) -> pd.DataFrame:
    """
    Returns a minimal, uniform schema WITHOUT adding 'id'/'source'.
    Columns:
      <row_id_col>, ISBN_clean (as 'isbn'), title_raw, author_raw, title_clean, author_clean
    """
    if "ISBN_clean" not in df.columns:
        raise KeyError(f"{prefix}: Expected 'ISBN_clean' column.")

    rid_col = row_id_colname or f"{prefix}_row_id"
    if rid_col not in df.columns:
        df = add_row_id(df, prefix=prefix, colname=rid_col)

    tcol = _pick(df, title_cols)
    acol = _pick(df, author_cols)
    if not tcol:
        raise KeyError(f"{prefix}: Could not find title column among {title_cols}")
    if not acol:
        raise KeyError(f"{prefix}: Could not find author column among {author_cols}")

    out = pd.DataFrame({
        rid_col: df[rid_col].astype(str),
        "isbn": df["ISBN_clean"].astype(str),
        "title_raw": df[tcol],
        "author_raw": df[acol],
    })
    out["title_clean"]  = out["title_raw"].map(_norm_text)
    out["author_clean"] = out["author_raw"].map(_norm_text)

    # drop duplicates by row id (keeps your stable IDs intact)
    out = out.drop_duplicates(subset=[rid_col]).reset_index(drop=True)
    return out, rid_col  # return the name of the id column for later functions

amazon=add_row_id(amazon,"amazon")
metabooks=add_row_id(metabooks,"metabooks")
goodreads=add_row_id(goodreads,"goodreads")

In [17]:
A, A_id = canonicalize_books_rows(
    amazon,
    prefix="amazon",
    title_cols=["title"],
    author_cols=["Book-Author","author"],
    row_id_colname="amazon_row_id"   # if you already created it; else omit
)

M, M_id = canonicalize_books_rows(
    metabooks,
    prefix="metabooks",
    title_cols=["title"],
    author_cols=["Author_Name","author"],
    row_id_colname="metabooks_row_id"
)

G, G_id = canonicalize_books_rows(
    goodreads,
    prefix="goodreads",
    title_cols=["title"],
    author_cols=["author","Author","Author_Name"],
    row_id_colname="goodreads_row_id"
)


In [18]:
display(A.head(1))
display(M.head(1))
display(G.head(1))

,amazon_row_id,isbn,title_raw,author_raw,title_clean,author_clean
0,amazon_1,0195153448,Classical Mythology,Mark P. O. Morford,classical mythology,mark p. o. morford


,metabooks_row_id,isbn,title_raw,author_raw,title_clean,author_clean
0,metabooks_1,1844255247,Alvis: The Story of the Red Triangle,Kenneth Day,alvis: the story of the red triangle,kenneth day


,goodreads_row_id,isbn,title_raw,author_raw,title_clean,author_clean
0,goodreads_1,0439023483,The Hunger Games,Suzanne Collins,the hunger games,suzanne collins


In [ ]:
def pair_random_nonmatches(df_left: pd.DataFrame, left_id_col: str,
                           df_right: pd.DataFrame, right_id_col: str,
                           n: int, seed: int = 42) -> pd.DataFrame:
    """
    Randomly sample non-matching pairs (avoid equal ISBNs when both present).
    Returns: id_left, id_right, label=0
    """
    rng = np.random.default_rng(seed)
    L = df_left[[left_id_col, "isbn"]].to_numpy()
    R = df_right[[right_id_col, "isbn"]].to_numpy()

    out, seen = [], set()
    BATCH = min(10000, max(1000, n * 5))

    while len(out) < n and len(seen) < len(L) * len(R):
        li = rng.integers(0, len(L), size=BATCH)
        ri = rng.integers(0, len(R), size=BATCH)
        for i, j in zip(li, ri):
            lid, lisbn = L[i]
            rid, risbn = R[j]
            if pd.notna(lisbn) and pd.notna(risbn) and str(lisbn) == str(risbn):
                continue  # likely true match
            key = (lid, rid)
            if key in seen:
                continue
            seen.add(key)
            out.append(key)
            if len(out) >= n:
                break

    return pd.DataFrame(out, columns=["id_left", "id_right"]).assign(label=0)


def pair_exact_matches(df_left: pd.DataFrame, left_id_col: str,
                       df_right: pd.DataFrame, right_id_col: str) -> pd.DataFrame:
    """
    Exact matches via ISBN equality.
    Returns: id_left, id_right, label=1
    """
    L = df_left.dropna(subset=["isbn"])[[left_id_col, "isbn"]].rename(
        columns={left_id_col: "id_left", "isbn": "isbn_l"}
    )
    R = df_right.dropna(subset=["isbn"])[[right_id_col, "isbn"]].rename(
        columns={right_id_col: "id_right", "isbn": "isbn_r"}
    )

    exact = (
        L.merge(R, left_on="isbn_l", right_on="isbn_r", how="inner")
         .loc[:, ["id_left", "id_right"]]
         .drop_duplicates()
         .assign(label=1)
    )
    return exact

def pair_random_nonmatches(df_left: pd.DataFrame, left_id_col: str,
                           df_right: pd.DataFrame, right_id_col: str,
                           n: int, seed: int = 42) -> pd.DataFrame:
    """
    Randomly sample non-matching pairs (avoid equal ISBNs when both present).
    Returns: id_left, id_right, label=0
    """
    rng = np.random.default_rng(seed)
    L = df_left[[left_id_col, "isbn"]].to_numpy()
    R = df_right[[right_id_col, "isbn"]].to_numpy()

    out, seen = [], set()
    BATCH = min(10000, max(1000, n * 5))

    while len(out) < n and len(seen) < len(L) * len(R):
        li = rng.integers(0, len(L), size=BATCH)
        ri = rng.integers(0, len(R), size=BATCH)
        for i, j in zip(li, ri):
            lid, lisbn = L[i]
            rid, risbn = R[j]
            if pd.notna(lisbn) and pd.notna(risbn) and str(lisbn) == str(risbn):
                continue  # likely true match
            key = (lid, rid)
            if key in seen:
                continue
            seen.add(key)
            out.append(key)
            if len(out) >= n:
                break

    return pd.DataFrame(out, columns=["id_left", "id_right"]).assign(label=0)



def _block_key(title: str, author: str) -> str:
    # very cheap block to reduce pairs; feel free to refine later
    return " ".join(title.split()[:2] + author.split()[:1])

def _pair_sim(ta, aa, tb, ab) -> float:
    st = fuzz.token_set_ratio(ta, tb) / 100.0
    sa = fuzz.token_set_ratio(aa, ab) / 100.0
    return 0.7 * st + 0.3 * sa

def pair_corner_candidates(df_left: pd.DataFrame, left_id_col: str,
                           df_right: pd.DataFrame, right_id_col: str,
                           band_low: float = 0.70, band_high: float = 0.85) -> pd.DataFrame:
    """
    Surface borderline pairs based on title+author similarity within [band_low, band_high].
    Returns a review sheet:
      id_left, id_right, title_left, author_left, title_right, author_right, sim, why, label(NaN)
    """
    L = df_left.copy()
    R = df_right.copy()
    L["blk"] = L.apply(lambda r: _block_key(r["title_clean"], r["author_clean"]), axis=1)
    R["blk"] = R.apply(lambda r: _block_key(r["title_clean"], r["author_clean"]), axis=1)

    C = L.merge(R, on="blk", suffixes=("_l", "_r"))
    if C.empty:
        return pd.DataFrame(columns=[
            "id_left","id_right","title_left","author_left","title_right","author_right","sim","why","label"
        ])

    C["sim"] = C.apply(lambda r: _pair_sim(r["title_clean_l"], r["author_clean_l"],
                                           r["title_clean_r"], r["author_clean_r"]), axis=1)
    C = C[(C["sim"] >= band_low) & (C["sim"] <= band_high)].copy()
    C["why"] = "text_boundary"

    review = (
        C.rename(columns={f"{left_id_col}":"id_left", f"{right_id_col}":"id_right"})
         .loc[:, ["id_left","id_right",
                  "title_raw_l","author_raw_l","title_raw_r","author_raw_r","sim","why"]]
         .rename(columns={
             "title_raw_l":"title_left","author_raw_l":"author_left",
             "title_raw_r":"title_right","author_raw_r":"author_right",
         })
         .drop_duplicates(subset=["id_left","id_right"])
         .assign(label=pd.NA)
         .sort_values("sim", ascending=False)
         .reset_index(drop=True)
    )
    return review




In [20]:
# Example for Amazon ↔ Metabooks
pos_MA  = pair_exact_matches(A, A_id, M, M_id)                 # label=1
corn_MA = pair_corner_candidates(A, A_id, M, M_id, 0.7, 0.85) # to label manually later
neg_MA  = pair_random_nonmatches(A, A_id, M, M_id, n=500, seed=42)  # label=0

# Example for Metabooks ↔ Goodreads
pos_MG  = pair_exact_matches(M, M_id, G, G_id)
corn_MG = pair_corner_candidates(M, M_id, G, G_id, 0.7, 0.85)
neg_MG  = pair_random_nonmatches(M, M_id, G, G_id, n=500, seed=42)


In [22]:
def compose_20_30_50(
    df_left: pd.DataFrame, left_id_col: str,
    df_right: pd.DataFrame, right_id_col: str,
    total_n: int,
    *,
    corner_band=(0.07, 0.85),
    seed: int = 42,
):
    """
    Build a labeled set with the classic 20/30/50 distribution:
      - 20% positives (exact ISBN matches)
      - 30% corners (text-similarity band; unlabeled -> to be labeled manually)
      - 50% negatives (random pairs avoiding equal ISBNs)
    Returns: (pos_df, corners_df, neg_df)
      pos_df: id_left, id_right, label=1
      corners_df: id_left, id_right, title_left, author_left, title_right, author_right, sim, why, label=NA
      neg_df: id_left, id_right, label=0
    """
    n_pos = int(round(total_n * 0.20))
    n_cor = int(round(total_n * 0.30))
    n_neg = total_n - n_pos - n_cor

    # 1) full pool of positives, then sample
    pos_pool = pair_exact_matches(df_left, left_id_col, df_right, right_id_col)
    pos = pos_pool.sample(n=min(n_pos, len(pos_pool)), random_state=seed) if len(pos_pool) else pos_pool

    # 2) full set of corners (banded), then sample
    low, high = corner_band
    corners_pool = pair_corner_candidates(df_left, left_id_col, df_right, right_id_col,
                                          band_low=low, band_high=high)
    corners = corners_pool.sample(n=min(n_cor, len(corners_pool)), random_state=seed) if len(corners_pool) else corners_pool

    # 3) sample negatives directly
    neg = pair_random_nonmatches(df_left, left_id_col, df_right, right_id_col, n=n_neg, seed=seed)

    return pos.reset_index(drop=True), corners.reset_index(drop=True), neg.reset_index(drop=True)


In [23]:
# Creating 1000 examples (Training- Validation- Test)

# Amazon ↔ Metabooks
pos_MA, corners_MA, neg_MA = compose_20_30_50(
    A, A_id, M, M_id,
    total_n=1000,
    corner_band=(0.70, 0.85),
    seed=42
)
# Metabooks - Goodreads
pos_MG, corners_MG, neg_MG =compose_20_30_50(
    M,M_id,A,A_id,1000,    corner_band=(0.70, 0.85),   
    seed=42
)

In [ ]:
import os
import pandas as pd

def export_triplets(df: pd.DataFrame, filename: str):
    """
    Write triplets with NO header:
        id_left,id_right,label
    Saves automatically inside ./sets/
    Assumes 'label' is 0/1 (no NaNs).
    """
    # Ensure ./sets folder exists
    os.makedirs("sets", exist_ok=True)
    path = os.path.join("sets", filename)

    required = {"id_left", "id_right", "label"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"export_triplets: missing columns {sorted(missing)}")

    # Enforce 0/1 integers (raise if NaN)
    if df["label"].isna().any():
        raise ValueError("export_triplets: 'label' contains NaN; only finalized 0/1 labels are allowed.")

    T = df[["id_left", "id_right", "label"]].copy()
    T["label"] = T["label"].astype(int)
    T.to_csv(path, index=False, header=False)
    print(f"✅ Triplets exported to: {path}")


def export_corner_review(
    corners_df: pd.DataFrame,
    filename: str,
    sample_n: int | None = None,
    seed: int = 42,
):
    """
    Export a review sheet for manual labeling with columns:
      id_left,id_right,title_left,author_left,title_right,author_right,sim,why,label
    - If sample_n is None or >= len(data), exports ALL rows.
    - Otherwise, exports a simple random sample (no stratification).
    Saves automatically inside ./sets/
    """
    import os
    os.makedirs("sets", exist_ok=True)
    path = os.path.join("sets", filename)

    cols = [
        "id_left","id_right",
        "title_left","author_left",
        "title_right","author_right",
        "sim","why","label"
    ]
    S = corners_df.copy()


    needed = {"id_left","id_right","title_left","author_left","title_right","author_right"}
    missing_needed = needed - set(S.columns)
    if missing_needed:
        raise KeyError(f"export_corner_review: missing columns {sorted(missing_needed)}")

    for c in ["sim","why","label"]:
        if c not in S.columns:
            S[c] = None

    if len(S) == 0:
        # nothing to export
        pd.DataFrame(columns=cols).to_csv(path, index=False)
        print(f"⚠️ No data to export — created empty file at {path}")
        return

    if sample_n is None or sample_n >= len(S):
        S[cols].to_csv(path, index=False)
    else:
        S.sample(n=sample_n, random_state=seed)[cols].to_csv(path, index=False)

    print(f"✅ Corner review sheet exported to: {path}")


In [ ]:
export_triplets(pos_MA, "neg_MA.csv")
export_triplets(neg_MA, "pos_MA.csv")
export_corner_review(corners_MA, "corners_MA_review.csv", sample_n=300, seed=42)
export_triplets(pos_MG, "neg_MG.csv")
export_triplets(neg_MG, "pos_MG.csv")
export_corner_review(corners_MG, "corners_MG_review.csv", sample_n=300, seed=42)

In [25]:
def finalize_and_split(pos_df, corners_labeled_df, neg_df, *, val_frac=0.2, test_frac=0.2, seed=42):
    """
    Merge positives + labeled corners + negatives, then stratified split.
    Returns: train_df, val_df, test_df  (all with id_left,id_right,label)
    """
    all_labeled = pd.concat(
        [pos_df[["id_left","id_right","label"]],
         corners_labeled_df[["id_left","id_right","label"]],
         neg_df[["id_left","id_right","label"]]],
        ignore_index=True
    ).dropna(subset=["label"])

    all_labeled["label"] = all_labeled["label"].astype(int)
    all_labeled = all_labeled.drop_duplicates()

    # stratified split
    L = all_labeled.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    parts = {"train": [], "val": [], "test": []}
    for y in (0, 1):
        grp = L[L["label"] == y]
        n = len(grp)
        n_test = int(round(n * test_frac))
        n_val  = int(round(n * val_frac))
        test_part = grp.iloc[:n_test]
        val_part  = grp.iloc[n_test:n_test+n_val]
        train_part= grp.iloc[n_test+n_val:]
        parts["train"].append(train_part)
        parts["val"].append(val_part)
        parts["test"].append(test_part)

    train = pd.concat(parts["train"], ignore_index=True)
    val   = pd.concat(parts["val"],   ignore_index=True)
    test  = pd.concat(parts["test"],  ignore_index=True)
    return train, val, test


In [ ]:
# Manually label the corners and then split all examples into train/val/test
labeled_corners_MG = pd.read_csv("sets/corners_MG_review.csv")
train_MG, val_MG, test_MG =finalize_and_split(pos_MG,neg_MG,labeled_corners_MG,val_frac=0.2,test_frac=0.2,seed=42)

In [ ]:
# Doing the same for other pair
labeled_corners_MA=pd.read_csv("sets/corners_MA_review.csv")
train_MA, val_MA, test_MA =finalize_and_split(pos_MA,neg_MA,labeled_corners_MA,val_frac=0.2,test_frac=0.2,seed=42)

In [ ]:
display(train_MG.label.describe())
display(val_MG.label.describe())
display(test_MG.label.describe())
# Stratified split

count    600.000000
mean       0.205000
std        0.404038
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: label, dtype: float64

count    200.000000
mean       0.205000
std        0.404715
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: label, dtype: float64

count    200.000000
mean       0.205000
std        0.404715
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: label, dtype: float64

In [35]:
train_MG.to_csv("sets/train_MG.csv",index=False)
val_MG.to_csv("sets/val_MG.csv",index=False)
test_MG.to_csv("sets/test_MG.csv",index=False)